# 211. Sparse Autoencoder：怎样从 LLM Activation 学稀疏 Feature？

> **面试问题：怎样实现 SAE 编码、top-k 稀疏化、解码重建、死特征统计和因果干预，并避免把相关性误读为可解释性？**

## 先给结论

不要只背论文名或框架名。应当说明输入/状态合同、核心算法、失败分支、独立 oracle、指标和可回滚制品。以下均使用受控小数据验证实现机制；真实生产仍需替换模型、权限、索引、安全审计与线上评测。

## 一手资料

- [Towards Monosemanticity](https://arxiv.org/abs/2309.08600)
- [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html)
- [Representation Engineering](https://arxiv.org/abs/2310.01405)

In [ ]:
contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "versioned"}  # 执行本行的状态、计算或校验逻辑。
assert contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert contract["production"] == "versioned"  # 执行本行的状态、计算或校验逻辑。
assert len(contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. Activation 与字典合同

SAE 的输入必须固定到特定 checkpoint、层、hook、归一化和采样集。宽 feature 字典将 dense activation 投影到更多候选特征；feature id 脱离这些版本就没有稳定语义。


In [ ]:
acts = [(2.0, 0.0), (0.0, 2.0), (1.0, 1.0)]  # 执行本行的状态、计算或校验逻辑。
encoder = ((1.0, 0.0), (0.0, 1.0), (1.0, 1.0))  # 执行本行的状态、计算或校验逻辑。
decoder = ((1.0, 0.0, 0.5), (0.0, 1.0, 0.5))  # 执行本行的状态、计算或校验逻辑。
assert len(acts) == 3  # 执行本行的状态、计算或校验逻辑。
assert len(encoder) == 3  # 执行本行的状态、计算或校验逻辑。
assert len(decoder) == 2  # 执行本行的状态、计算或校验逻辑。


## 2. 编码与非负激活

编码器线性投影后常使用 ReLU 或类似门控，便于把 feature 激活视为强度。真实 SAE 会有 bias/归一化等细节；这里保留显式矩阵计算，方便检查形状和数值。


In [ ]:
def matvec(matrix, vector):  # 执行本行的状态、计算或校验逻辑。
    return tuple(sum(row[i] * vector[i] for i in range(len(vector))) for row in matrix)  # 执行本行的状态、计算或校验逻辑。
def encode(vector):  # 执行本行的状态、计算或校验逻辑。
    return tuple(max(0.0, value) for value in matvec(encoder, vector))  # 执行本行的状态、计算或校验逻辑。
features = encode(acts[0])  # 执行本行的状态、计算或校验逻辑。
assert features == (2.0, 0.0, 2.0)  # 执行本行的状态、计算或校验逻辑。
assert all(value >= 0 for value in features  # 执行本行的状态、计算或校验逻辑。
            )  # 执行本行的状态、计算或校验逻辑。
assert len(features) == 3  # 执行本行的状态、计算或校验逻辑。


## 3. 显式稀疏化

重建误差小并不意味着 feature 稀疏或单义。top-k gate 是易于测试的稀疏合同：每个样本至多 k 个活跃特征；生产中应同 L1/jump-ReLU 等方案在同一评测集对照。


In [ ]:
def top_k(values, k):  # 执行本行的状态、计算或校验逻辑。
    order = sorted(range(len(values)), key=lambda i: values[i], reverse=True)[:k]  # 执行本行的状态、计算或校验逻辑。
    return tuple(value if i in order else 0.0 for i, value in enumerate(values))  # 执行本行的状态、计算或校验逻辑。
sparse = top_k(features, 2)  # 执行本行的状态、计算或校验逻辑。
assert sum(value > 0 for value in sparse) == 2  # 执行本行的状态、计算或校验逻辑。
assert sparse == (2.0, 0.0, 2.0)  # 执行本行的状态、计算或校验逻辑。
assert top_k((1.0, 3.0, 2.0), 1) == (0.0, 3.0, 0.0)  # 执行本行的状态、计算或校验逻辑。


## 4. 解码与重建

decoder 将 sparse feature 组合回 activation 空间。重建是必要条件，却不是可解释性的充分条件：低 reconstruction loss 仍可能来自多个 feature 共同编码复杂模式。


In [ ]:
def decode(values):  # 执行本行的状态、计算或校验逻辑。
    return tuple(sum(decoder[r][c] * values[c] for c in range(len(values))) for r in range(len(decoder))  # 执行本行的状态、计算或校验逻辑。
                 )  # 执行本行的状态、计算或校验逻辑。
def squared_error(left, right):  # 执行本行的状态、计算或校验逻辑。
    return sum((a - b) ** 2 for a, b in zip(left, right))  # 执行本行的状态、计算或校验逻辑。
reconstruction = decode(sparse)  # 执行本行的状态、计算或校验逻辑。
assert reconstruction == (3.0, 1.0)  # 执行本行的状态、计算或校验逻辑。
assert squared_error(reconstruction, acts[0]) == 2.0  # 执行本行的状态、计算或校验逻辑。
assert len(reconstruction) == 2  # 执行本行的状态、计算或校验逻辑。


## 5. 损失取舍

SAE 训练常组合 reconstruction loss 和稀疏惩罚。稀疏权重越高不一定越好：可能减少 feature 使用却损伤重建和下游行为，应以 Pareto 曲线而非单一数字选择超参。


In [ ]:
def objective(act, rec, feat, weight):  # 执行本行的状态、计算或校验逻辑。
    return squared_error(act, rec) + weight * sum(abs(value) for value in feat)  # 执行本行的状态、计算或校验逻辑。
low = objective(acts[0], reconstruction, sparse, 0.01)  # 执行本行的状态、计算或校验逻辑。
high = objective(acts[0], reconstruction, sparse, 1.0)  # 执行本行的状态、计算或校验逻辑。
assert low > 0  # 执行本行的状态、计算或校验逻辑。
assert high > low  # 执行本行的状态、计算或校验逻辑。
assert objective((0.0, 0.0), (0.0, 0.0), (0.0, 0.0), 1.0) == 0.0  # 执行本行的状态、计算或校验逻辑。


## 6. 死特征监控

宽字典会出现从未激活的 dead feature。要在固定 activation 样本上统计活跃频次，并把死特征率、平均 L0、重建误差及训练稳定性共同报告；不能只展示几个好看的例子。


In [ ]:
def usage(data):  # 执行本行的状态、计算或校验逻辑。
    counts = [0, 0, 0]  # 执行本行的状态、计算或校验逻辑。
    for act in data:  # 执行本行的状态、计算或校验逻辑。
        for i, value in enumerate(top_k(encode(act), 2)):  # 执行本行的状态、计算或校验逻辑。
            counts[i] += int(value > 0)  # 执行本行的状态、计算或校验逻辑。
    return counts  # 执行本行的状态、计算或校验逻辑。
counts = usage(acts)  # 执行本行的状态、计算或校验逻辑。
assert counts == [2, 1, 3]  # 执行本行的状态、计算或校验逻辑。
assert sum(value == 0 for value in counts) == 0  # 执行本行的状态、计算或校验逻辑。
assert len(counts) == 3  # 执行本行的状态、计算或校验逻辑。


## 7. 干预而非相关性

查看高激活样本只能提出 feature 命名假设。更强的证据来自对特征消融/放大后，独立 verifier 观察到的目标行为变化与无关任务保持；务必限制可干预的层和版本。


In [ ]:
def intervene(values, index, replacement):  # 执行本行的状态、计算或校验逻辑。
    updated = list(values)  # 执行本行的状态、计算或校验逻辑。
    updated[index] = replacement  # 执行本行的状态、计算或校验逻辑。
    return tuple(updated)  # 执行本行的状态、计算或校验逻辑。
ablated = intervene(sparse, 0, 0.0)  # 执行本行的状态、计算或校验逻辑。
assert ablated[0] == 0.0  # 执行本行的状态、计算或校验逻辑。
assert ablated[2] == sparse[2]  # 执行本行的状态、计算或校验逻辑。
assert decode(ablated) != reconstruction  # 执行本行的状态、计算或校验逻辑。


## 8. 制品与回归

发布 SAE 时保存 base model、层、hook、字典参数、稀疏规则、训练样本、重建/死特征指标和干预评测。否则 feature 编号可能在重训后漂移，审查和回归无法复放。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"model": "demo-v1", "layer": 12, "hook": "residual", "features": 3, "top_k": 2}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["features"] == 3  # 执行本行的状态、计算或校验逻辑。
assert artifact["top_k"] == 2  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整答案应先定义成功条件，再说明数据状态、主路径、失败边界和评测。受控断言只证明实现不变量，不能直接外推为真实大语料、模型语义、线上成本或安全效果。
